# 01 — Prepare PALM designs 

## Paths

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().resolve().parent
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "output"
PALM_DIR = OUTPUT_DIR / "voxel_statistics" / "palm"
PALM_DIR.mkdir(parents=True, exist_ok=True)

FLNM_DIR = DATA_DIR / "processed" / "fLNM_2mm_leaddbs"
LESION_DIR = DATA_DIR / "raw" / "lesion_masks" / "acuteinfarct_2mm"

def flnm_path(sub):
    return f"{FLNM_DIR}/{sub}_infarct_conn-GSP1000v2xFullSet_desc-T_funcmap.nii.gz"

def lesion_path(sub):
    return f"{LESION_DIR}/{sub}_infarct.nii.gz"

## Load clinical data

In [ ]:
clinical = pd.read_csv(DATA_DIR / "clean" / "clinical_data.csv")

DOMAINS = ["aef", "ps", "language", "verbalmemory", "vsfunctions", "vsmemory"]
DOMAIN_COLS = {d: f"cognition-{d}_domain_score" for d in DOMAINS}
VOL_COL = "stroke-total_infarct_volume"
NIHSS_COL = "stroke-NIHSS"
COHORT_COL = "demographics-cohort"

clinical[VOL_COL] = pd.to_numeric(clinical[VOL_COL], errors="coerce")
clinical[NIHSS_COL] = pd.to_numeric(clinical[NIHSS_COL], errors="coerce")
for c in DOMAIN_COLS.values():
    clinical[c] = pd.to_numeric(clinical[c], errors="coerce")

## NIHSS imputation

In [ ]:
clinical["nihss_imputed"] = clinical[NIHSS_COL].isna().astype(int)

cohort_median = clinical.groupby(COHORT_COL)[NIHSS_COL].transform("median")
global_median = clinical[NIHSS_COL].median()

nihss_imp = clinical[NIHSS_COL].fillna(cohort_median)
fell_back_global = nihss_imp.isna()  # cohorts with no NIHSS at all
nihss_imp = nihss_imp.fillna(global_median)
clinical["stroke-NIHSS_imp"] = nihss_imp

n_imp = int(clinical["nihss_imputed"].sum())
assert clinical["stroke-NIHSS_imp"].notna().all()

## Writer helper

In [ ]:
manifest = []  # collect a summary row per variant

def write_variant(out_dir, sub_df, design_cols, contrast, image_path_fn, eb_codes=None):
    """Write design/contrast/input_paths (+ optional eb.csv) for one PALM variant.

    sub_df       : rows already filtered + null-dropped, in final order
    design_cols  : list of column names in sub_df forming the design matrix
    contrast     : list of floats, len == len(design_cols)
    image_path_fn: sub_id -> absolute image path string
    eb_codes     : optional integer block code per subject (within-cohort exchangeability
                   blocks); writes a single-column eb.csv consumed by PALM `-eb ... -within`.
                   Group comparisons pass this; the pure one-sample arm passes None.
    """
    assert len(contrast) == len(design_cols), (out_dir, len(contrast), len(design_cols))
    out_dir.mkdir(parents=True, exist_ok=True)
    sub_df[design_cols].to_csv(out_dir / "design.csv", header=False, index=False)
    pd.DataFrame([contrast]).to_csv(out_dir / "contrast.csv", header=False, index=False)
    paths = sub_df["sub_id"].map(image_path_fn)
    paths.to_csv(out_dir / "input_paths.txt", header=False, index=False)
    eb_path = out_dir / "eb.csv"
    n_eb = 0
    if eb_codes is not None:
        codes = np.asarray(eb_codes)
        pd.DataFrame({"eb": codes}).to_csv(eb_path, header=False, index=False)
        n_eb = int(pd.Series(codes).nunique())
    elif eb_path.exists():
        eb_path.unlink()
    n_missing = int((~paths.map(lambda p: Path(p).exists())).sum())
    rel = out_dir.relative_to(PALM_DIR)
    manifest.append({
        "variant": str(rel), "n_subjects": len(sub_df),
        "n_design_cols": len(design_cols), "n_eb_blocks": n_eb,
        "contrast": str(contrast), "n_images_missing": n_missing,
    })
    return len(sub_df), n_missing, n_eb

# constant columns used in designs
clinical["intercept"] = 1

## Covariate specification

In [ ]:
# layer name -> extra covariate columns (beyond the group dummy).
# design is always [group] + extra_cov + cohort_dummies + [intercept];
# contrast tests the group coefficient only: [1] + 0*(everything else).
COV_LAYERS = {
    "volcov":   [VOL_COL],
    "volnihss": [VOL_COL, "stroke-NIHSS_imp"],
    "nocov":    [],                      # no lesion-volume covariate (cohort FE + EB still kept)
}

# Which covariate layers each image family receives:
#   LNM  -> nocov (primary) + volnihss (volume + NIHSS sensitivity)
#   VLSM -> volcov only
# Every group variant — including nocov — retains the cohort fixed effects and the within-cohort
# exchangeability blocks.
FAMILY_LAYERS = {
    "lnm":  ["nocov", "volnihss"],
    "vlsm": ["volcov"],
}

def build_group_family(family, image_path_fn):
    """Build the group-comparison variants for one image family.

    Multi-cohort confounding is handled two ways, together:
      * cohort FIXED EFFECTS  -> k-1 cohort dummies in the design (reference cohort handled
        by the intercept); removes additive site differences from the t-map.
      * within-cohort EXCHANGEABILITY BLOCKS -> eb.csv (one integer per subject) so PALM
        permutes the symptom label only WITHIN each cohort (-eb eb.csv -within).
    The one-sample arm gets neither (kept pure).
    """
    for layer in FAMILY_LAYERS[family]:
        extra_cov = COV_LAYERS[layer]
        for d in DOMAINS:
            gcol = DOMAIN_COLS[d]
            need = ["sub_id", gcol, COHORT_COL] + extra_cov
            sub = clinical.dropna(subset=need).copy()
            sub[gcol] = sub[gcol].astype(int)

            # cohort fixed effects (drop_first -> reference cohort goes into the intercept)
            dummies = pd.get_dummies(sub[COHORT_COL], prefix="cohort", drop_first=True).astype(int)
            for col in dummies.columns:
                sub[col] = dummies[col].values
            cohort_cols = list(dummies.columns)

            # within-cohort exchangeability blocks (1-based integer code per subject)
            eb_codes = pd.factorize(sub[COHORT_COL])[0] + 1

            design_cols = [gcol] + extra_cov + cohort_cols + ["intercept"]
            contrast = [1] + [0] * (len(design_cols) - 1)
            out_dir = PALM_DIR / f"{family}_{layer}" / f"cognition-{d}_domain_score"
            n, miss, neb = write_variant(out_dir, sub, design_cols, contrast,
                                         image_path_fn, eb_codes=eb_codes)

build_group_family("lnm", flnm_path)
build_group_family("vlsm", lesion_path)

## One-sample LNM (impaired cases only)

In [ ]:
# Design is a single intercept column; contrast `[1]`. Input images are the fLNM T-funcmaps of the impaired cases.
for d in DOMAINS:
    gcol = DOMAIN_COLS[d]
    sub = clinical[clinical[gcol] == 1].dropna(subset=["sub_id"]).copy()
    out_dir = PALM_DIR / "onesample" / f"cognition-{d}_domain_score"
    n, miss, _ = write_variant(out_dir, sub, ["intercept"], [1], flnm_path)

## Sanity summary

In [ ]:
manifest_df = pd.DataFrame(manifest)
manifest_df.to_csv(PALM_DIR / "_manifest.csv", index=False)